# Introspection Factorization — demo**Six cells.** Cells 1–2 and 6 need no GPU and run in about a minute: they pull theartifacts and regenerate every figure and number in the paper. Cells 3–5 loadQwen3.6-27B and run a live injection, and need an A100 80 GB.The claim: a concept can be *present* in the residual stream and *readable* by theJacobian lens while the model says nothing about it. Cell 4 shows both halves ofthat sentence at once.

In [ ]:
# 1 — install + artifacts.  No GPU needed for cells 1, 2, 6.%pip -q install numpy scipy pandas matplotlib pyyaml diptest huggingface_hub%pip -q install "transformers>=4.57.1" accelerate%pip -q install "git+https://github.com/anthropics/jacobian-lens@581d398613e5602a5af361e1c34d3a92ea82ba8e"import os, sys, json, subprocessfrom pathlib import PathREPO_URL = os.environ.get("REPO_URL", "")          # set to your git remote, or upload the folderARTIFACTS_REPO = os.environ.get("ARTIFACTS_REPO", "")  # HF dataset id holding artifacts/REPO = Path("/content/introspection-factorization")if not REPO.exists():    if REPO_URL:        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)    else:        for c in (Path.cwd(), Path.cwd().parent):            if (c / "src" / "factors.py").exists():                REPO = c; breakos.chdir(REPO); sys.path.insert(0, str(REPO / "src"))if ARTIFACTS_REPO:    from huggingface_hub import snapshot_download    snapshot_download(ARTIFACTS_REPO, repo_type="dataset",                      local_dir=str(REPO / "artifacts"))    print("artifacts pulled from", ARTIFACTS_REPO)else:    print("ARTIFACTS_REPO not set -- using whatever is in ./artifacts")    print("to mirror your own run:")    print("  huggingface-cli upload <you>/introspection-factorization artifacts --repo-type=dataset")have = (REPO / "artifacts" / "factors" / "factors_input.npz").exists()print(f"\nrepo    {REPO}\nfactors {'present' if have else 'MISSING -- cells 2 and 6 need it'}")

In [ ]:
# 2 — regenerate all three figures from artifacts.  No GPU.!python scripts/05_figures.py --k 10from IPython.display import Image, displayfor name in ("fig1_cascade", "fig2_order_contrast", "fig3_f2_sensitivity"):    p = Path("artifacts/figures") / f"{name}.png"    if p.exists():        display(Image(filename=str(p)))    else:        print("missing", p)

In [ ]:
# 3 — load the model and the pre-fitted lens.  GPU from here on (A100 80 GB).import torch, yaml, transformers, jlensimport lens as lens_mod, inject, vectors as vec_mod, prompts as prompt_modcfg = yaml.safe_load(open("configs/sprint.yaml"))tok = transformers.AutoTokenizer.from_pretrained(    cfg["model"]["repo"], revision=cfg["model"]["revision"])hf = transformers.AutoModelForImageTextToText.from_pretrained(    cfg["model"]["repo"], revision=cfg["model"]["revision"],    dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa")model = jlens.from_hf(hf, tok)lens = lens_mod.load_lens(cfg["lens"]["repo"], cfg["lens"]["filename"],                          cfg["lens"]["revision"], device=model.input_device)LAYER = cfg["planned"]["layers"][0]baseline_words, _ = vec_mod.load_baseline_words("configs/baseline_words.json")print(f"model {cfg['model']['repo']}  |  lens layers {len(lens.source_layers)}  "      f"|  injecting at layer {LAYER} ({cfg['planned']['layer_block_type']})")

In [ ]:
# 4 — LIVE: inject a concept, then read the lens beside what the model says.CONCEPT = "Bread"          # <-- type any single-token conceptALPHA_REL = 4.0vecs, _ = vec_mod.extract_concept_vectors(model, tok, [CONCEPT], baseline_words, LAYER)ids = model.encode(prompt_mod.render(tok, prompt_mod.detect_messages(), prefill=True))stop = int(ids.shape[1]) - 4positions = slice(stop - 8, stop)norm = inject.median_residual_norm(model, ids, LAYER, positions)out = inject.injected_prefill(    model, ids, LAYER, torch.stack([vecs[CONCEPT]]).to(model.input_device),    torch.tensor([ALPHA_REL * norm], device=model.input_device), positions,    record_layers=[LAYER], record_positions=[-1])top_ids, _ = lens_mod.readout(model, lens, out["residuals"][LAYER], LAYER, k=10)print(f"J-lens readout at the report position (layer {LAYER}):")print("   ", [tok.decode([t]) for t in top_ids[0]])gen = inject.generate_with_injection(    model, hf, tok, ids, LAYER, torch.stack([vecs[CONCEPT]]).to(model.input_device),    torch.tensor([ALPHA_REL * norm], device=model.input_device), positions,    max_new_tokens=64, temperature=1.0, seed=0)print(f"\nWhat the model actually says:\n    {gen['completions'][0].strip()[:300]!r}")print(f"\ninjected: {CONCEPT!r}   hook fired {gen['n_fires']}x (prefill only)")

In [ ]:
# 5 — order toggle: identical injection, both orders, side by side.import sweep as sweep_modanswer = sweep_mod.clean_task_answer(model, hf, tok, prompt_mod.TASK_PROMPTS[0])for order in cfg["planned"]["orders"]:    oids, opos, _ = sweep_mod.sweep_prompt(        model, tok, order, prompt_mod.TASK_PROMPTS[0], answer, 8, 4)    onorm = inject.median_residual_norm(model, oids, LAYER, opos)    v = torch.stack([vecs[CONCEPT]]).to(model.input_device)    a = torch.tensor([ALPHA_REL * onorm], device=model.input_device)    r = inject.injected_prefill(model, oids, LAYER, v, a, opos,                                record_layers=[LAYER], record_positions=[-1])    tids, _ = lens_mod.readout(model, lens, r["residuals"][LAYER], LAYER, k=5)    g = inject.generate_with_injection(model, hf, tok, oids, LAYER, v, a, opos,                                       max_new_tokens=48, temperature=1.0, seed=0)    print(f"--- {order} ---")    print("  lens top-5 :", [tok.decode([t]) for t in tids[0]])    print("  model says :", g["completions"][0].strip().replace("\n", " ")[:200])    print()

In [ ]:
# 6 — the full cascade, from artifacts.  No GPU.import jsonr = json.load(open("artifacts/figures/results.json"))c, ci = r["cascade"], r["cascade_ci"]print(f"operating point: layer {r['operating_point']['layer']}, "      f"alpha_rel {r['operating_point']['strength']}, k={r['operating_point']['k']}")print(f"n = {r['n_trials']} injection trials over {r['n_concepts']} concepts\n")for name, label in (("f1", "P(represented)"),                    ("f2", "P(verbalizable | represented)"),                    ("f3", "P(reported | verbalizable)")):    p, lo, hi = ci[name]    print(f"  {name}  {label:<32} {p:.3f}  [{lo:.3f}, {hi:.3f}]")print(f"\n  product   {c['f1']*c['f2']*c['f3']:.4f}")print(f"  observed  {c['observed_cascade_rate']:.4f}")print(f"  cascade_residual {c['residual']:.3e}   <- must be ~0, or a denominator is wrong")print(f"\n  survivorship  {c['n_entering_f1']} -> {c['n_surviving_f1']} -> "      f"{c['n_surviving_f2']} -> {c['n_surviving_f3']}")print(f"\n  losses:  representation {1-c['f1']:.3f}   "      f"verbalizability {1-c['f2']:.3f}   channel closure {1-c['f3']:.3f}")print(open("RESULTS.md").read())